inline_ret_0 low
inline_ret_1 low
inline_ret_2 low
inline_ret_3 low
inline_ret_4 low
amp_0 high
lower_shadow_0 high
lower_shadow_1 high
lower_shadow_2 high

In [ ]:
import optuna
import polars as pl
from zh import (
    CLOSE,
    HIGH,
    LOW,
    OPEN,
    OPEN_TIME,
    SHARPE_DAY,
    SYMBOL,
    UTC,
    kline_plot,
    plot,
    scan,
)

data = scan("1m")

hold = 5
fees = 10e-4
start_date = pl.datetime(2023, 1, 1, time_zone=UTC)
train_date = pl.datetime(2024, 1, 1, time_zone=UTC)
val_date = pl.datetime(2025, 1, 1, time_zone=UTC)
test_date = pl.datetime(2025, 12, 31, time_zone=UTC)

In [ ]:
train_data = (
    data.filter(OPEN_TIME >= start_date)
    .with_columns(
        lower_shadow_0=((pl.min_horizontal(OPEN, CLOSE) - LOW) / OPEN),
        lower_shadow_1=((pl.min_horizontal(OPEN, CLOSE) - LOW) / OPEN)
        .shift(1)
        .over(SYMBOL),
        lower_shadow_2=((pl.min_horizontal(OPEN, CLOSE) - LOW) / OPEN)
        .shift(2)
        .over(SYMBOL),
        amp_0=(HIGH - LOW) / OPEN,
        inline_ret_0=(CLOSE / OPEN - 1),
        inline_ret_1=(CLOSE / OPEN - 1).shift(1).over(SYMBOL),
        inline_ret_2=(CLOSE / OPEN - 1).shift(2).over(SYMBOL),
        inline_ret_3=(CLOSE / OPEN - 1).shift(3).over(SYMBOL),
        inline_ret_4=(CLOSE / OPEN - 1).shift(4).over(SYMBOL),
        ret=((CLOSE.shift(1 - hold) / OPEN - 1 - fees) / hold).shift(-1).over(SYMBOL),
    )
    .collect(engine="streaming")
)


In [ ]:
def get_rets(
    data: pl.LazyFrame | pl.DataFrame,
    condition,
    start_date=start_date,
    end_date=train_date,
) -> pl.LazyFrame:
    return (
        data.lazy()
        .filter(OPEN_TIME.is_between(start_date, end_date))
        .select(OPEN_TIME, SYMBOL, pl.when(condition).then(pl.col("ret")))
        .group_by(OPEN_TIME)
        .agg((pl.col("ret").mean()).alias("ret"))
        .fill_null(0)
        .sort(OPEN_TIME)
    )


In [ ]:
def objective(trial: optuna.Trial) -> float:
    inline_ret_0_max = trial.suggest_float("inline_ret_0_max", -0.01, 0.01)
    inline_ret_1_max = trial.suggest_float("inline_ret_1_max", -0.01, 0.01)
    inline_ret_2_max = trial.suggest_float("inline_ret_2_max", -0.01, 0.01)
    inline_ret_3_max = trial.suggest_float("inline_ret_3_max", -0.01, 0.01)
    inline_ret_4_max = trial.suggest_float("inline_ret_4_max", -0.01, 0.01)
    amp_0_min = trial.suggest_float("amp_0_min", 0.0, 0.01)
    lower_ratio_0_min = trial.suggest_float("lower_ratio_0_min", 0.0, 0.01)
    lower_ratio_1_min = trial.suggest_float("lower_ratio_1_min", 0.0, 0.01)
    lower_ratio_2_min = trial.suggest_float("lower_ratio_2_min", 0.0, 0.01)

    condition = (
        (pl.col("inline_ret_0") <= inline_ret_0_max)
        & (pl.col("inline_ret_1") <= inline_ret_1_max)
        & (pl.col("inline_ret_2") <= inline_ret_2_max)
        & (pl.col("inline_ret_3") <= inline_ret_3_max)
        & (pl.col("inline_ret_4") <= inline_ret_4_max)
        & (pl.col("amp_0") >= amp_0_min)
        & (pl.col("lower_shadow_0") >= lower_ratio_0_min)
        & (pl.col("lower_shadow_1") >= lower_ratio_1_min)
        & (pl.col("lower_shadow_2") >= lower_ratio_2_min)
    )

    sharpe = (
        get_rets(
            train_data,
            condition,
            start_date=start_date,
        )
        .group_by_dynamic(OPEN_TIME, every="1d")
        .agg(pl.col("ret").sum().alias("ret"))
        .select(SHARPE_DAY)
        .fill_nan(-1000)
        .collect(engine="streaming")
        .item()
    )
    return -sharpe


study = optuna.create_study()
study.optimize(objective, n_trials=3000)

In [ ]:
print(study.best_value, study.best_params)

In [ ]:
params = {
    "inline_ret_0_max": 0.008965891766013027,
    "inline_ret_1_max": -0.005308940497450236,
    "inline_ret_2_max": 0.0012446653865547106,
    "inline_ret_3_max": 0.009554799461824413,
    "inline_ret_4_max": 0.008958378924030543,
    "amp_0_min": 0.008085414094176896,
    "lower_ratio_0_min": 0.003913081942498345,
    "lower_ratio_1_min": 0.0003959007913899283,
    "lower_ratio_2_min": 4.0289311027941705e-06,
}

condition = (
    (pl.col("inline_ret_0") <= params["inline_ret_0_max"])
    & (pl.col("inline_ret_1") <= params["inline_ret_1_max"])
    & (pl.col("inline_ret_2") <= params["inline_ret_2_max"])
    & (pl.col("inline_ret_3") <= params["inline_ret_3_max"])
    & (pl.col("inline_ret_4") <= params["inline_ret_4_max"])
    & (pl.col("amp_0") >= params["amp_0_min"])
    & (pl.col("lower_shadow_0") >= params["lower_ratio_0_min"])
    & (pl.col("lower_shadow_1") >= params["lower_ratio_1_min"])
    & (pl.col("lower_shadow_2") >= params["lower_ratio_2_min"])
)

In [ ]:
params = {
    "inline_ret_0_max": 0.0063947406079792645,
    "inline_ret_1_max": 0.008473434063249593,
    "inline_ret_2_max": -0.004297104790732571,
    "inline_ret_3_max": 0.00924902963775518,
    "inline_ret_4_max": 0.004622435570302349,
    "amp_0_min": 0.0006126429631741897,
    "lower_ratio_0_min": 0.003551762163610814,
    "lower_ratio_1_min": 0.00018485874598347506,
    "lower_ratio_2_min": 0.0004989474645579084,
}

condition = (
    (pl.col("inline_ret_0") <= params["inline_ret_0_max"])
    & (pl.col("inline_ret_1") <= params["inline_ret_1_max"])
    & (pl.col("inline_ret_2") <= params["inline_ret_2_max"])
    & (pl.col("inline_ret_3") <= params["inline_ret_3_max"])
    & (pl.col("inline_ret_4") <= params["inline_ret_4_max"])
    & (pl.col("amp_0") >= params["amp_0_min"])
    & (pl.col("lower_shadow_0") >= params["lower_ratio_0_min"])
    & (pl.col("lower_shadow_1") >= params["lower_ratio_1_min"])
    & (pl.col("lower_shadow_2") >= params["lower_ratio_2_min"])
)

In [ ]:
params = {
    "inline_ret_0_max": 0.008205699967523085,
    "inline_ret_1_max": 0.005311961312860898,
    "inline_ret_2_max": -0.004591329678260734,
    "inline_ret_3_max": 0.007906767977432687,
    "inline_ret_4_max": 0.007465783345942942,
    "amp_0_min": 0.007443766207905541,
    "lower_ratio_0_min": 0.0038840964086050596,
    "lower_ratio_1_min": 0.00017533577986860783,
    "lower_ratio_2_min": 0.0004395684411790212,
}

condition = (
    (pl.col("inline_ret_0") <= params["inline_ret_0_max"])
    & (pl.col("inline_ret_1") <= params["inline_ret_1_max"])
    & (pl.col("inline_ret_2") <= params["inline_ret_2_max"])
    & (pl.col("inline_ret_3") <= params["inline_ret_3_max"])
    & (pl.col("inline_ret_4") <= params["inline_ret_4_max"])
    & (pl.col("amp_0") >= params["amp_0_min"])
    & (pl.col("lower_shadow_0") >= params["lower_ratio_0_min"])
    & (pl.col("lower_shadow_1") >= params["lower_ratio_1_min"])
    & (pl.col("lower_shadow_2") >= params["lower_ratio_2_min"])
)

In [ ]:
params = {
    "inline_ret_0_max": 0.008899979237197176,
    "inline_ret_1_max": 0.009473724865957626,
    "inline_ret_2_max": -0.00453610789456007,
    "inline_ret_3_max": 0.006076457675630603,
    "inline_ret_4_max": 0.006218649810554277,
    "amp_0_min": 0.003215234977431744,
    "lower_ratio_0_min": 0.004039215342775666,
    "lower_ratio_1_min": 0.002510791988133569,
    "lower_ratio_2_min": 0.0005418162315184008,
}

condition = (
    (pl.col("inline_ret_0") <= params["inline_ret_0_max"])
    & (pl.col("inline_ret_1") <= params["inline_ret_1_max"])
    & (pl.col("inline_ret_2") <= params["inline_ret_2_max"])
    & (pl.col("inline_ret_3") <= params["inline_ret_3_max"])
    & (pl.col("inline_ret_4") <= params["inline_ret_4_max"])
    & (pl.col("amp_0") >= params["amp_0_min"])
    & (pl.col("lower_shadow_0") >= params["lower_ratio_0_min"])
    & (pl.col("lower_shadow_1") >= params["lower_ratio_1_min"])
    & (pl.col("lower_shadow_2") >= params["lower_ratio_2_min"])
)

In [ ]:
params = {
    "inline_ret_0_max": 0.008678149698720855,
    "inline_ret_1_max": 0.007581411553462681,
    "inline_ret_2_max": -0.004245388406024142,
    "inline_ret_3_max": 0.008958195388751521,
    "inline_ret_4_max": 0.007159462637936492,
    "amp_0_min": 0.006267524502763932,
    "lower_ratio_0_min": 0.004166125672562209,
    "lower_ratio_1_min": 6.28914245086403e-05,
    "lower_ratio_2_min": 6.0950217804062615e-06,
}

condition = (
    (pl.col("inline_ret_0") <= params["inline_ret_0_max"])
    & (pl.col("inline_ret_1") <= params["inline_ret_1_max"])
    & (pl.col("inline_ret_2") <= params["inline_ret_2_max"])
    & (pl.col("inline_ret_3") <= params["inline_ret_3_max"])
    & (pl.col("inline_ret_4") <= params["inline_ret_4_max"])
    & (pl.col("amp_0") >= params["amp_0_min"])
    & (pl.col("lower_shadow_0") >= params["lower_ratio_0_min"])
    & (pl.col("lower_shadow_1") >= params["lower_ratio_1_min"])
    & (pl.col("lower_shadow_2") >= params["lower_ratio_2_min"])
)

In [ ]:
condition = (
    (pl.col("inline_ret_0") <= study.best_params["inline_ret_0_max"])
    & (pl.col("inline_ret_1") <= study.best_params["inline_ret_1_max"])
    & (pl.col("inline_ret_2") <= study.best_params["inline_ret_2_max"])
    & (pl.col("inline_ret_3") <= study.best_params["inline_ret_3_max"])
    & (pl.col("inline_ret_4") <= study.best_params["inline_ret_4_max"])
    & (pl.col("amp_0") >= study.best_params["amp_0_min"])
    & (pl.col("lower_shadow_0") >= study.best_params["lower_ratio_0_min"])
    & (pl.col("lower_shadow_1") >= study.best_params["lower_ratio_1_min"])
    & (pl.col("lower_shadow_2") >= study.best_params["lower_ratio_2_min"])
)

In [ ]:
plot(
    get_rets(
        train_data,
        condition,
        start_date=pl.datetime(2020, 1, 1, time_zone=UTC),
        end_date=test_date,
    )
)

In [ ]:
get_rets(
    train_data, condition, start_date=val_date, end_date=test_date, fees=3e-4
).group_by_dynamic(OPEN_TIME, every="1d").agg(pl.col("ret").sum()).select(
    SHARPE_DAY
).collect(engine="streaming").item()

In [ ]:
kline_plot(data=train_data, condition=condition, pre=4, post=10)